# papikapi-style LoRA — FLUX.2 klein 4B

Trains the `papikapi-style` LoRA on the base checkpoint `black-forest-labs/FLUX.2-klein-base-4B`.
The resulting LoRA is loaded on the distilled `FLUX.2-klein-4B` used by the studio.

**Before running**

1. Settings → Accelerator → **GPU T4 x2**.
2. Settings → **Internet on** (needs a phone-verified Kaggle account).
3. Add Input → upload `papikapi-dataset.zip` as a Kaggle dataset (54 `.jpeg` + 54 `.txt`).
4. Run All. Expect roughly 3–4 hours on a T4 (bf16 is emulated there).

## 1. GPU check

In [ ]:
import subprocess

query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if query.returncode != 0 or not query.stdout.strip():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2 (or L4), then restart the session.")

gpu_name, compute_cap = [field.strip() for field in query.stdout.splitlines()[0].split(",")]
TRAIN_DTYPE = "bf16"
print(gpu_name, f"sm_{compute_cap}", "->", TRAIN_DTYPE)

## 2. Install ai-toolkit

In [ ]:
!git clone https://github.com/ostris/ai-toolkit /kaggle/working/ai-toolkit
!cd /kaggle/working/ai-toolkit && git checkout ecee894ed2b1f3716d9d7326693061ec1a3105bb && git submodule update --init --recursive
!pip install -q -r /kaggle/working/ai-toolkit/requirements.txt

## 3. Dataset

In [ ]:
import shutil
from pathlib import Path

captions = sorted(Path("/kaggle/input").rglob("*.txt"))
if not captions:
    raise FileNotFoundError("No .txt captions under /kaggle/input. Add papikapi-dataset as an input.")

dataset_dir = Path("/kaggle/working/dataset")
shutil.rmtree(dataset_dir, ignore_errors=True)
shutil.copytree(captions[0].parent, dataset_dir)

images = sorted(dataset_dir.glob("*.jpeg"))
print(f"{len(images)} images, {len(list(dataset_dir.glob('*.txt')))} captions")
print(Path(images[0]).with_suffix(".txt").read_text())

## 4. Train

In [ ]:
import json
from collections import OrderedDict

TRAIN_STEPS = 1000
SAMPLE_EVERY = 250

SAMPLE_PROMPTS = [
    "papikapi-style, a standing penguin",
    "papikapi-style, a sitting white rabbit",
    "papikapi-style, a red sports car",
    "papikapi-style, a standing panda cub",
]

job = OrderedDict([
    ("job", "extension"),
    ("config", OrderedDict([
        ("name", "papikapi_style_v1"),
        ("process", [OrderedDict([
            ("type", "diffusion_trainer"),
            ("training_folder", "/kaggle/working/output"),
            ("device", "cuda:0"),
            ("network", OrderedDict([("type", "lora"), ("linear", 16), ("linear_alpha", 16)])),
            ("save", OrderedDict([("dtype", "bf16"), ("save_every", SAMPLE_EVERY), ("max_step_saves_to_keep", 6)])),
            ("datasets", [OrderedDict([
                ("folder_path", str(dataset_dir)),
                ("caption_ext", "txt"),
                ("caption_dropout_rate", 0.05),
                ("cache_latents_to_disk", True),
                ("resolution", [512]),
            ])]),
            ("train", OrderedDict([
                ("batch_size", 1),
                ("steps", TRAIN_STEPS),
                ("gradient_accumulation_steps", 1),
                ("train_unet", True),
                ("train_text_encoder", False),
                ("content_or_style", "style"),
                ("gradient_checkpointing", True),
                ("noise_scheduler", "flowmatch"),
                ("timestep_type", "weighted"),
                ("optimizer", "adamw8bit"),
                ("lr", 1e-4),
                ("dtype", TRAIN_DTYPE),
            ])),
            ("model", OrderedDict([
                ("name_or_path", "black-forest-labs/FLUX.2-klein-base-4B"),
                ("arch", "flux2_klein_4b"),
                ("quantize", True),
                ("qtype", "qfloat8"),
                ("quantize_te", True),
                ("low_vram", True),
                ("model_kwargs", {"match_target_res": False}),
            ])),
            ("sample", OrderedDict([
                ("sampler", "flowmatch"),
                ("sample_every", SAMPLE_EVERY),
                ("width", 512),
                ("height", 512),
                ("prompts", SAMPLE_PROMPTS),
                ("neg", ""),
                ("seed", 42),
                ("walk_seed", True),
                ("guidance_scale", 4),
                ("sample_steps", 20),
            ])),
        ])]),
    ])),
    ("meta", OrderedDict([("name", "papikapi_style_v1"), ("version", "1.0")])),
])

config_file = Path("/kaggle/working/papikapi_style_v1.json")
config_file.write_text(json.dumps(job, indent=2))
!cd /kaggle/working/ai-toolkit && HF_HUB_ENABLE_HF_TRANSFER=1 python run.py {config_file}

## 5. Package the result

Download `papikapi-lora-output.zip` from the Output panel. It holds every saved LoRA checkpoint and the
sample images. Pick the checkpoint whose samples look best, which is often not the last one.

In [ ]:
!cd /kaggle/working/output && zip -q -r /kaggle/working/papikapi-lora-output.zip papikapi_style_v1 -x "*/.aitk*"
!ls -lh /kaggle/working/papikapi-lora-output.zip /kaggle/working/output/papikapi_style_v1/*.safetensors
if not list(Path("/kaggle/working/output/papikapi_style_v1").glob("*.safetensors")):
    raise RuntimeError("Training produced no LoRA checkpoint. See the training cell output above.")

from PIL import Image, ImageStat
latest_sample = max(Path("/kaggle/working/output/papikapi_style_v1/samples").glob("*.jpg"), key=lambda path: path.stat().st_mtime)
brightness = ImageStat.Stat(Image.open(latest_sample).convert("L")).mean[0]
print(f"Latest sample {latest_sample.name} brightness {brightness:.1f}")
if brightness < 5:
    raise RuntimeError("Samples are black: the model produced invalid values (numeric overflow).")